# AWS Bedrock AgentCore - Execute Command 데모

이 Notebook에서는 다음 방법을 살펴봅니다.
1. Bedrock AgentCore agent 생성 및 배포
2. 표준 prompt로 에이전트 호출
3. [`invoke_agent_runtime_command`](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/invoke_agent_runtime_command.html)를 사용하여 agent runtime에서 system command 직접 실행

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Runtime에서 command 실행                                  |
| Tool 유형           | HTTP server                                               |
| 튜토리얼 구성 요소  | AgentCore Runtime에 호스팅                                |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 중간                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK                        |


## 1단계: Dependency 설치

uv package manager를 사용하여 필요한 Python package를 설치합니다.

## 빠른 시작 가이드

**이 Notebook 실행 방법:**
1. 위에서 아래로 셀을 순서대로 실행합니다.
2. 에이전트 파일을 생성한 후(2단계) **kernel을 재시작**합니다.
3. 3단계부터 계속 진행합니다.

**학습 내용:**
- Jupyter Notebook에서 Bedrock AgentCore agent를 배포하는 방법
- 여러 방법으로 에이전트를 호출하는 방법
- **agent runtime에서 shell command를 직접 실행하는 방법** ⭐

In [ ]:
!uv pip install -Uq -r requirements.txt

**⚠️ 중요**

- `pip install cell`을 실행한 뒤 library가 올바르게 설치되도록 kernel을 재시작합니다.
- `invoke_agent_runtime_command`는 boto3 version `1.42.69`**`에서 도입되었으므로 적절한 version을 사용하고 있는지 확인합니다.

In [ ]:
!uv pip freeze | grep -i boto3

## 2단계: 에이전트 코드 정의

에이전트 entry point 파일을 생성합니다. 이 에이전트는 다음 항목을 사용합니다.
- **BedrockAgentCoreApp**: AgentCore 애플리케이션을 빌드하는 framework
- **Strands Agent**: user prompt를 처리하는 AI agent

In [ ]:
%%writefile agents/agent.py
# Bedrock AgentCore에 필요한 라이브러리 가져오기
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent

# AgentCore 애플리케이션 초기화
app = BedrockAgentCoreApp()

# AI agent instance 생성
agent = Agent()

@app.entrypoint
def invoke(payload, context):
    """
    에이전트의 기본 엔트리포인트입니다.
    
    매개변수:
        payload: 사용자 입력이 담긴 'prompt' 키를 포함하는 딕셔너리
        context: Runtime context 정보
    
    반환값:
        에이전트 응답 메시지가 포함된 딕셔너리
    """
    # payload에서 user prompt 추출
    user_message = payload.get("prompt", "Hello!")
    
    # 에이전트로 message 처리
    result = agent(user_message)
    
    # 예상 형식으로 응답 반환
    return {"result": result.message}

if __name__ == "__main__":
    # agent 애플리케이션 실행
    app.run()

In [ ]:
%%writefile agents/requirements.txt
bedrock-agentcore
strands-agents

## 3단계: AWS 구성 설정

AWS client를 초기화하고 에이전트 배포에 필요한 계정 정보를 가져옵니다.

In [ ]:
import boto3
from boto3.session import Session

# 기본 credentials로 boto3 session 초기화
boto_session = Session()

# AWS 계정 정보 가져오기
sts = boto3.client("sts")
response = sts.get_caller_identity()
account_id = response["Account"]
region = boto_session.region_name

print(f"AWS Account ID: {account_id}")
print(f"AWS Region: {region}")

# 에이전트 호출용 Bedrock AgentCore client 생성
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

## 4단계: 에이전트 구성 및 배포

에이전트 배포 설정을 구성합니다. toolkit은 다음 작업을 처리합니다.
- IAM execution role 자동 생성
- container image용 ECR repository 설정
- Python 코드에서 직접 배포(Docker 불필요)

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Runtime toolkit 초기화
agentcore_runtime_agent = Runtime()

# 에이전트의 고유 이름 정의
aws_agent_name = "exec_cmd_sample"

# 다음 설정으로 에이전트 배포 구성:
# - entrypoint: 에이전트 코드 파일 경로
# - auto_create_execution_role: 필요한 권한이 있는 IAM role 자동 생성
# - auto_create_ecr: container image용 ECR repository 자동 생성
# - requirements_file: Python dependency 경로
# - region: 배포할 AWS 리전
# - agent_name: 에이전트의 고유 identifier
# - protocol: 통신 protocol(REST 형식 상호 작용에는 HTTP 사용)
# - deployment_type: Docker 없이 Python 코드 직접 배포
# - runtime_type: runtime 환경의 Python version
response_aws_agent = agentcore_runtime_agent.configure(
    entrypoint="agents/agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_agent_name,
    protocol="HTTP",
    deployment_type="direct_code_deploy",
    runtime_type="PYTHON_3_13",
)

print("Configuration completed:", response_aws_agent)

In [ ]:
# AWS에 에이전트 배포
# 이 과정에서 다음 작업을 수행함:
# 1. IAM execution role 생성 또는 업데이트
# 2. 에이전트 코드와 dependency package 생성
# 3. S3에 업로드
# 4. Bedrock AgentCore Runtime에 배포
# 5. CloudWatch Logs 및 X-Ray tracing 구성
launch_result = agentcore_runtime_agent.launch()

print("Launch completed:", launch_result.agent_arn)

# 이후 호출을 위해 agent ARN 저장
cmd_agent_arn = launch_result.agent_arn

In [ ]:
# 배포 상태 확인
# 에이전트를 호출하려면 "READY" 상태여야 함
status_response = agentcore_runtime_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

## 5단계: 에이전트 호출 테스트

다음 두 가지 방법으로 배포된 에이전트를 테스트합니다.
1. **High-level SDK 방식**: toolkit의 간소화된 invoke 방식 사용
2. **직접 boto3 방식**: 호출을 더 세밀하게 제어하도록 AWS SDK 사용

In [ ]:
# 방식 1: high-level toolkit 방식으로 호출
# 에이전트를 호출하는 가장 간단한 방법
invoke_response = agentcore_runtime_agent.invoke({"prompt": "What can u do?"})
invoke_response["response"]

## 6단계: System Command 실행

**핵심 기능**: agent runtime 환경에서 임의의 system command를 직접 실행합니다.

`invoke_agent_runtime_command`는 다음 기능을 제공합니다.
- 에이전트의 containerized runtime에서 shell command 실행
- stdout/stderr output 실시간 streaming
- exit code 및 실행 상태 반환
- 에이전트 환경의 debugging, 파일 작업, script 실행에 활용

In [ ]:
# agent runtime에서 system command 실행
# Command: /tmp 디렉터리의 파일을 상세 정보와 함께 나열
response = agentcore_client.invoke_agent_runtime_command(
    agentRuntimeArn=cmd_agent_arn,
    body={
        "command": '/bin/bash -c "ls -l /tmp"',  # 실행할 shell command
        "timeout": 300,  # timeout 초 단위(5분)
    },
)

# command output stream 처리
for event in response["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        print(chunk)

## 7단계: 리소스 정리(선택 사항)

지속적인 요금이 발생하지 않도록 AWS 리소스를 정리합니다. 이 작업은 다음 리소스를 삭제합니다.
- Bedrock AgentCore Runtime

**⚠️ 경고**: 이 작업은 되돌릴 수 없습니다. 테스트를 마친 경우에만 실행하세요.

In [ ]:
from pathlib import Path
from bedrock_agentcore_starter_toolkit.operations.runtime.destroy import (
    destroy_bedrock_agentcore,
)

print("🚀 Starting Runtime cleanup...")

# agent runtime 및 관련 리소스 모두 삭제
# .bedrock_agentcore.yaml 파일에서 구성 읽기
destroy_bedrock_agentcore(config_path=Path(".bedrock_agentcore.yaml"), agent_name=aws_agent_name)

print("✅ Cleanup completed successfully!")

---

## 요약

이 Notebook에서는 다음 작업을 살펴봤습니다.

1. ✅ **에이전트 생성**: Python 코드로 Bedrock AgentCore agent 배포
2. ✅ **에이전트 호출**: high-level 및 low-level 방식으로 에이전트 호출
3. ✅ **Command 실행**: `invoke_agent_runtime_command`를 사용해 runtime 환경에서 shell command 실행

### 핵심 요점

- **Event Streaming**: `invoke_agent_runtime_command`는 세 가지 event type으로 구성된 event stream을 반환합니다.
  - `contentStart`: command 실행이 시작되었음을 나타냄
  - `contentDelta`: streaming stdout/stderr output 포함
  - `contentStop`: 최종 exit code와 상태 제공
  
- **사용 사례**: 이 기능은 다음 작업에 유용합니다.
  - 진단 command 실행
  - 데이터 처리 script 실행
  - 에이전트 환경의 파일 시스템 작업
  - 통합 테스트 및 debugging